# Hands-on 35: Studi Kasus A - Smart Room Monitoring Lengkap

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 14.

Sistem lengkap: sensor/simulator -> MQTT -> subscriber Python (validasi, fitur
lag/rolling) -> klasifikasi nyaman/tidak_nyaman -> storage. Label mengikuti aturan buku:
nyaman jika suhu 24-28 C, kelembaban 45-65%, CO2 < 1000.

## 2. Arsitektur

Arsitektur minimal yang dipakai:

sensor / simulator → MQTT → Python subscriber → preprocessing → model klasifikasi → InfluxDB → Grafana

Komponen:

- simulator Python atau sensor nyata
- Mosquitto MQTT
- Python subscriber untuk parsing, preprocessing, inferensi
- InfluxDB untuk penyimpanan time series
- Grafana untuk dashboard

Arsitektur ini konsisten dengan stack open source buku ini: ESP32/simulator, Mosquitto, Python, InfluxDB/SQLite, dan Grafana.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /downloads/iot-ml-book


## 1. Latih Model (baseline LR vs Random Forest)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/case_a_train_room_model.py"], capture_output=True, text=True)
print(r.stdout)

[INFO] Distribusi label: {'tidak_nyaman': 1400, 'nyaman': 758}

=== Baseline Logistic Regression (accuracy): 0.6551 ===

=== Random Forest - Confusion Matrix ===
[[150   2]
 [  0 280]]

=== Random Forest - Classification Report ===
              precision    recall  f1-score   support

      nyaman       1.00      0.99      0.99       152
tidak_nyaman       0.99      1.00      1.00       280

    accuracy                           1.00       432
   macro avg       1.00      0.99      0.99       432
weighted avg       1.00      1.00      1.00       432

[SUKSES] Model disimpan ke: models/case_a_room_rf.joblib
[SUKSES] Laporan disimpan ke: outputs/reports/case_a_evaluation.csv



## 2. Deployment Mini (broker + subscriber + publisher)

In [3]:
influx_hapus("case_a_room")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/case_a_room_subscriber.py", "case_a_sub_nb.log")
    time.sleep(2)
    pub = latar("scripts/case_a_room_publisher.py", "case_a_pub_nb.log")
    time.sleep(24)   # sekitar 12 pesan
    henti(pub, sub)
    print(log_teks("case_a_sub_nb.log"))
finally:
    broker.terminate()

Subscribe ke casea/room01/telemetry
[2026-11-02T12:19:04] -> status=tidak_nyaman
[2026-11-02T12:19:06] -> status=tidak_nyaman
[2026-11-02T12:19:08] -> status=nyaman
[2026-11-02T12:19:10] -> status=nyaman
[2026-11-02T12:19:12] -> status=nyaman
[2026-11-02T12:19:14] -> status=tidak_nyaman
[2026-11-02T12:19:16] -> status=tidak_nyaman
[2026-11-02T12:19:18] -> status=nyaman
[2026-11-02T12:19:20] -> status=tidak_nyaman
[2026-11-02T12:19:22] -> status=tidak_nyaman
[2026-11-02T12:19:24] -> status=tidak_nyaman
[2026-11-02T12:19:26] -> status=tidak_nyaman



## 3. Verifikasi Hasil

In [4]:
df = influx_df("case_a_room")
print(df.to_string(index=False))
print("\nDistribusi status:", df["pred_status"].value_counts().to_dict() if "pred_status" in df else "")

                    _time device_id  pred_status  co2_ppm  humidity_pct  temperature_c
2026-11-02 12:19:04+00:00    room01 tidak_nyaman    849.0         52.45          29.88
2026-11-02 12:19:06+00:00    room01 tidak_nyaman   1498.0         65.93          28.10
2026-11-02 12:19:08+00:00    room01       nyaman   1278.0         53.99          25.65
2026-11-02 12:19:10+00:00    room01       nyaman   1357.0         50.24          26.70
2026-11-02 12:19:12+00:00    room01       nyaman   1124.0         67.53          25.78
2026-11-02 12:19:14+00:00    room01 tidak_nyaman   1043.0         53.26          23.60
2026-11-02 12:19:16+00:00    room01 tidak_nyaman   1000.0         40.97          27.66
2026-11-02 12:19:18+00:00    room01       nyaman   1238.0         64.75          24.99
2026-11-02 12:19:20+00:00    room01 tidak_nyaman    954.0         65.12          30.57
2026-11-02 12:19:22+00:00    room01 tidak_nyaman    608.0         64.43          31.32
2026-11-02 12:19:24+00:00    room01 tidak_n

## Jawaban dari data: Fokus studi kasus A: jawaban dari data

In [5]:
import pandas as pd
ev = pd.read_csv("outputs/reports/case_a_evaluation.csv")
print(ev.to_string(index=False))
import joblib
from sklearn.metrics import confusion_matrix
m = joblib.load("models/case_a_room_rf.joblib")
d = pd.read_csv("data/synthetic/room_classification_dataset.csv")
d["status"] = ((d.temperature_c.between(24, 28)) & (d.humidity_pct.between(45, 65)) & (d.co2_ppm < 1000)).map({True: "nyaman", False: "tidak_nyaman"})
print("Distribusi label aturan:", d["status"].value_counts().to_dict())
print("Apakah tidak_nyaman sering lolos sebagai nyaman? Lihat recall/F1 tidak_nyaman di tabel di atas: F1 tidak_nyaman =", round(float(ev.loc[ev.model == "random_forest", "f1_tidak_nyaman"].iloc[0]), 4))

              model  accuracy  f1_nyaman  f1_tidak_nyaman
logistic_regression  0.655093        NaN              NaN
      random_forest  0.995370   0.993377         0.996441


Distribusi label aturan: {'tidak_nyaman': 1402, 'nyaman': 758}
Apakah tidak_nyaman sering lolos sebagai nyaman? Lihat recall/F1 tidak_nyaman di tabel di atas: F1 tidak_nyaman = 0.9964


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

### 1. Masalah

Tujuan studi kasus ini adalah membangun sistem yang dapat:

- membaca suhu, kelembaban, dan CO₂
- mengklasifikasikan kondisi ruangan menjadi nyaman atau tidak nyaman
- menampilkan hasil pada dashboard real-time

Kasus ini sangat realistis untuk:

- ruang kelas
- laboratorium komputer
- ruang kerja
- ruang server kecil
- greenhouse ringan

Referensi juga menunjukkan bahwa pada domain smart home dan smart building, sensor lingkungan seperti suhu, kelembaban, dan CO₂ sangat relevan untuk occupancy-aware control, efisiensi operasional, serta kesehatan dan keselamatan lingkungan.

### 3. Dataset

Opsi data nyata

Payload minimal: ● timestamp ● device_id ● temperature_c ● humidity_pct ● co2_ppm

Opsi synthetic fallback

Jika sensor belum ada, gunakan data sintetis ruangan dengan pola: ●   suhu harian ●   kelembaban berlawanan arah ringan dengan suhu ●   CO₂ meningkat pada jam sibuk ●   beberapa kondisi ekstrem untuk label tidak_nyaman Fallback sintetis ini penting karena studi kasus harus tetap bisa dijalankan walau perangkat belum lengkap, sesuai arah buku yang menekankan praktik yang repeatable.

### 4. Preprocessing

Langkah preprocessing yang dipakai:

Langkah 1 — validasi payload

Periksa: ● field wajib ada ● nilai numerik valid ● rentang wajar

Contoh rentang wajar: ● suhu: 10–45°C ● kelembaban: 0–100% ● CO₂: > 0

Langkah 2 — parsing waktu

● ubah timestamp ke datetime ● urutkan data

Langkah 3 — buat fitur

Fitur minimal: ● temperature_c ● humidity_pct ● co2_ppm ● hour ● is_weekend ● temp_lag_1 ● co2_lag_1 ● temp_ma_3 ● co2_ma_3

Langkah 4 — buat label

Aturan sederhana: ● nyaman jika suhu 24–28°C, kelembaban 45–65%, CO₂ < 1000 ● selain itu: tidak_nyaman

### 5. Model

Model yang paling realistis:

- Logistic Regression sebagai baseline
- Random Forest Classifier sebagai model utama

Kenapa Random Forest?

- cocok untuk data tabular
- cukup kuat untuk hubungan non-linear ringan
- feature importance mudah dijelaskan

### 6. Evaluasi

Metrik yang dipakai:

- accuracy
- precision
- recall
- F1
- confusion matrix

Fokus utama:

- apakah tidak_nyaman sering lolos sebagai nyaman
- apakah model terlalu sensitif memberi tidak_nyaman

### 7. Dashboard real-time

Panel Grafana yang disarankan:

- suhu real-time
- kelembaban real-time
- CO₂ real-time
- status ruangan (nyaman/tidak_nyaman)
- histori status
- panel alert sederhana

### 8.1. Simulator publisher MQTT

Buat file scripts/case_a_room_publisher.py:

```python
import json
import random
import time
from datetime import datetime
import paho.mqtt.client as mqtt
BROKER = "localhost"
TOPIC = "casea/room01/telemetry"
client = mqtt.Client(client_id="casea-room-pub")
client.connect(BROKER, 1883, 60)
while True:
    payload = {
        "timestamp":
datetime.now().isoformat(timespec="seconds"),
        "device_id": "room01",
        "temperature_c": round(random.uniform(23.0, 33.0), 2),
        "humidity_pct": round(random.uniform(40.0, 72.0), 2),
        "co2_ppm": random.randint(450, 1500)
    }
    client.publish(TOPIC, json.dumps(payload), qos=1)
    print("Published:", payload)
    time.sleep(2)
```

### 8.2. Subscriber + klasifikasi

Buat file scripts/case_a_room_subscriber.py:

```python
import json
from collections import deque
from pathlib import Path
import joblib
import pandas as pd
import paho.mqtt.client as mqtt
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
BROKER = "localhost"
TOPIC = "casea/room01/telemetry"
MODEL_PATH = Path("models/case_a_room_rf.joblib")
INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG = "iotlab"
INFLUX_BUCKET = "sensor_data"
temp_buffer = deque(maxlen=3)
co2_buffer = deque(maxlen=3)
model = joblib.load(MODEL_PATH)
client_db = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN,
org=INFLUX_ORG)
write_api = client_db.write_api(write_options=SYNCHRONOUS)
def safe_mean(values):
    return sum(values) / len(values) if len(values) > 0 else None
def build_features(payload):
    ts = pd.to_datetime(payload["timestamp"])
    temp = float(payload["temperature_c"])
    hum = float(payload["humidity_pct"])
    co2 = float(payload["co2_ppm"])
    temp_buffer.append(temp)
    co2_buffer.append(co2)
    return pd.DataFrame([{
        "temperature_c": temp,
        "humidity_pct": hum,
        "co2_ppm": co2,
        "hour": ts.hour,
        "is_weekend": int(ts.dayofweek >= 5),
        "temp_lag_1": temp_buffer[-2] if len(temp_buffer) >= 2 else
temp,
        "co2_lag_1": co2_buffer[-2] if len(co2_buffer) >= 2 else co2,
        "temp_ma_3": safe_mean(temp_buffer),
        "co2_ma_3": safe_mean(co2_buffer),
    }])
 def on_connect(client, userdata, flags, reason_code, properties=None):
     client.subscribe(TOPIC, qos=1)
     print("Subscribe ke", TOPIC)
 def on_message(client, userdata, msg):
     payload = json.loads(msg.payload.decode())
     X = build_features(payload)
     pred = model.predict(X)[0]
     print(f"[{payload['timestamp']}] -> status={pred}")
     point = (
         Point("case_a_room")
         .tag("device_id", payload["device_id"])
         .tag("pred_status", str(pred))
         .field("temperature_c", float(payload["temperature_c"]))
         .field("humidity_pct", float(payload["humidity_pct"]))
         .field("co2_ppm", float(payload["co2_ppm"]))
         .time(payload["timestamp"], WritePrecision.S)
     )
     write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG,
 record=point)
 client = mqtt.Client(client_id="casea-room-sub")
 client.on_connect = on_connect
 client.on_message = on_message
 client.connect(BROKER, 1883, 60)
 client.loop_forever()
```

Nilai praktis

Studi kasus ini cocok sebagai proyek mini untuk smart classroom, lab monitoring, dan ruang server ringan.

## Penutup

Studi kasus A menyatukan seluruh stack: Random Forest mencapai F1 sekitar 1.00 pada
label aturan kenyamanan, dan pipeline MQTT berjalan ujung ke ujung dengan storage CSV
(InfluxDB + Grafana opsional via `--influx`, panel: suhu, kelembaban, CO2, status,
histori status, alert). Model di `models/case_a_room_rf.joblib`.